# camo-eval Colab Demo

This notebook is the first public demo surface for `camo-eval`. It focuses on the stable Python API and batch evaluation workflow.

## 1. Install

For local development, replace the GitHub URL with a mounted workspace path.

In [ ]:
!pip install git+https://github.com/MichaelCSHN/CamoGED.git#subdirectory=camo-eval

## 2. Single-pair metrics

In [ ]:
import numpy as np
from camo_eval import (
    mae, s_measure, e_measure, weighted_f_measure, f_measure,
    precision, recall, precision_recall_curve, iou, dice, ssim
)

pred = np.array([[0, 255], [255, 0]], dtype=np.uint8)
gt = np.array([[0, 255], [255, 0]], dtype=np.uint8)

print('MAE:', mae(pred, gt))
print('Fw:', weighted_f_measure(pred, gt))
print('Sm:', s_measure(pred, gt))
print('Em:', e_measure(pred, gt))
print('F:', f_measure(pred, gt))
print('Precision:', precision(pred, gt))
print('Recall:', recall(pred, gt))
print('IoU:', iou(pred, gt))
print('Dice:', dice(pred, gt))
print('SSIM:', ssim(pred, gt))


## 3. Batch evaluation

In [ ]:
from pathlib import Path
from camo_eval import evaluate, to_markdown

pred_dir = Path('pred_demo')
gt_dir = Path('gt_demo')
pred_dir.mkdir(exist_ok=True)
gt_dir.mkdir(exist_ok=True)

np.save(pred_dir / 'sample.npy', pred)
np.save(gt_dir / 'sample.npy', gt)

results = evaluate(
    pred_dir, gt_dir,
    ['mae', 'fw', 'sm', 'em', 'f', 'precision', 'recall', 'iou', 'dice', 'ssim']
)
print(results)
print(to_markdown(results))


## 3b. Run the bundled repository demo dataset

In [ ]:
import json
from pathlib import Path

demo_root = Path('CamoGED/camo-eval/demo_data/cod_sota_masks')
manifest = json.loads((demo_root / 'manifest.json').read_text())
demo_results = evaluate(
    demo_root / manifest['pred_dir'],
    demo_root / manifest['gt_dir'],
    manifest['metrics']
)
print(to_markdown(demo_results))

## 3c. Visualize mask, error map, and PR curve


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from camo_eval.visualization import mask_overlay, error_map

sample_pred = demo_root / manifest['pred_dir'] / 'sample1.pgm'
sample_gt = demo_root / manifest['gt_dir'] / 'sample1.pgm'
pred_img = np.asarray(Image.open(sample_pred))
gt_img = np.asarray(Image.open(sample_gt))
curve = precision_recall_curve(pred_img, gt_img)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mask_overlay(pred_img, gt_img))
axes[0].set_title('mask overlay')
axes[1].imshow(error_map(pred_img, gt_img))
axes[1].set_title('error map')
axes[2].plot(curve['recall'], curve['precision'])
axes[2].set_title('PR curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].grid(alpha=0.3)
for ax in axes[:2]:
    ax.axis('off')
plt.tight_layout()


## 4. Protocol-aware reporting

In [ ]:
from camo_eval import EvaluationContext, EvaluationReport

context = EvaluationContext(
    observer='model',
    channel='rgb',
    task='image-cod',
    protocol='COD10K test split'
)
report = EvaluationReport(context=context, metrics=results.rows[0])
print(report.to_json())
print(report.to_markdown())